# **(Data Visualization)**

## Objectives

* Hypothesis 2: There is a visual difference between malignant and non-malignant average images
Validation process: Compute and display the average image for each class. If they appear visually different(like in colour or shape).

* Plot images for skin visualizer and hypothesis page

## Inputs

* inputs/skin_cancer_dataset/test
* inputs/skin_cancer_dataset/train
* inputs/skin_cancer_dataset/Validation

## Outputs

* outputs/v1/avg_var_benign.png
* outputs/v1/avg_var_malignant.png
* outputs/v1/benign_montage.png
* outputs/v1/malignant.png
* outputs/v1/image_sizes.pkl
* outputs/v1/benign_vs_malignant_differences.png
## Additional Comments

* No additional comments

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory
  
We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()


In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

---

# Data Directories

## Input Directories

Set train, validation, and test paths so machine knows where the data is

In [ ]:
my_data_dir = 'inputs/skin_cancer_dataset'
train_path = my_data_dir + '/train'
val_path = my_data_dir + '/validation'
test_path = my_data_dir + '/test'

## Output Directories

Set output directories so machine knows where to place the outputs

In [ ]:
version = 'v1'
file_path = f'outputs/{version}'

if 'outputs' in os.listdir(current_dir) and version in os.listdir(current_dir + '/outputs'):
    print('Old version is already available create a new version.')
    pass
else:
    os.makedirs(name=file_path)


## Set Label Names

In [ ]:
labels = os.listdir(train_path)
print('Label for the images are', labels)

## Import libraries to work with

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
sns.set_style("white")
from matplotlib.image import imread
import pickle
from tensorflow.keras.preprocessing import image

---

## Image Shape

See the average image size in the train set

In [ ]:
dim1 = []
dim2 = []

for label in labels:
    for image_filename in os.listdir(train_path + '/' + label):
        img = imread(train_path + '/' + label + '/' + image_filename)
        d1, d2, _ = img.shape
        dim1.append(d1)
        dim2.append(d2)

print(f'Average image size: {np.mean(dim2):.2f} x {np.mean(dim1):.2f} pixels (width x height)')
    
sns.set_style('white')
plt.figure(figsize=(12,6))
plt.title('Distribution of Image Dimensions')
plt.xlabel('Width')
plt.ylabel('Height')
plt.scatter(dim1, dim2)

In [ ]:
image_sizes = set()

for label in labels:
    for image_filename in os.listdir(train_path + '/' + label):
        image = imread(train_path + '/' + label + '/' + image_filename)
        image_sizes.add(image.shape[:2])

print(f'Number of unique image sizes: {len(image_sizes)}')
print(f'Image sizes found (height, width): {sorted(image_sizes)}')

if len(image_sizes) == 1:
    print('All training images have the exact same size.')
else:
    print('Training images do not all have the same size.')

As seen above, all images have the same size(224px x 224px). Now there is no need to change the dimensions of the images. Lastly, we can save the plot as a .pkl file

In [ ]:
joblib.dump(value=image_sizes, filename=f"{file_path}/image_sizes.pkl")

---

## Colour Format

Here, we will check the colour format and see if ti needs any changing

In [ ]:
from pathlib import Path
from PIL import Image


def convert_images_to_rgb(data_dir):
    supported_extensions = {'.jpg', '.jpeg', '.png'}
    converted_files = []

    for file_path in Path(data_dir).rglob('*'):
        if not file_path.is_file() or file_path.suffix.lower() not in supported_extensions:
            continue

        with Image.open(file_path) as image:
            if image.mode != 'RGB':
                image_format = image.format
                rgb_image = image.convert('RGB')
                rgb_image.save(file_path, format=image_format)
                converted_files.append(str(file_path))

    return converted_files


converted_files = convert_images_to_rgb('inputs/skin_cancer_dataset')
print(f'Converted {len(converted_files)} images to RGB.')

Here we can see that all images were already in RGB format, so there is no need to change this.

---

## Average and Variability of Images per Label

NOTE: In this section I will be using code from the walkthrough project

Function to load images into an array

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import os

def load_image_as_array(data_dir, new_size=(224, 224), n_images_per_label=30):
    X_list = []
    y_list = []

    labels = sorted(
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    )

    for label in labels:
        label_dir = os.path.join(data_dir, label)
        count = 0

        for image_filename in sorted(os.listdir(label_dir)):
            if count >= n_images_per_label:
                break

            img_path = os.path.join(label_dir, image_filename)

            if not os.path.isfile(img_path):
                continue

            img = image.load_img(img_path, target_size=new_size, color_mode="rgb")
            arr = image.img_to_array(img)

            # Normalize if pixel values are in 0..255
            if arr.max() > 1:
                arr = arr / 255.0

            X_list.append(arr)
            y_list.append(label)
            count += 1

    X = np.stack(X_list) if X_list else np.empty((0, *new_size, 3), dtype=np.float32)
    y = np.array(y_list, dtype=object)

    return X, y

Load image shape and labels in array

In [ ]:
new_size = next(iter(image_sizes))   # gives (224, 224)

X, y = load_image_as_array(
    data_dir=train_path,
    new_size=new_size,
    n_images_per_label=30
)

print(X.shape, y.shape)

Plot and save mean and variability of images per label

In [ ]:
def plot_mean_variability_per_labels(X, y, figsize=(12,5), save_image=False):
  """
   The pseudo code for the function is:
  * Loop in all labels
  * Subset an array for given label
  * Calculate mean and standard deviation
  * Create a figure displaying mean and variability of images
  * Save image

  """

  for label_to_display in np.unique(y):
    sns.set_style("white")

    y = y.reshape(-1,1,1)
    boolean_mask = np.any(y==label_to_display,axis=1).reshape(-1)
    arr = X[boolean_mask]

    avg_img = np.mean(arr, axis = 0)
    std_img = np.std(arr, axis = 0)
    print(f"==== Label {label_to_display} ====")
    print(f"Image Shape: {avg_img.shape}")
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=figsize)
    axes[0].set_title(f"Average Image for label {label_to_display}")
    axes[0].imshow(avg_img, cmap='gray')
    axes[1].set_title(f"Variability image for label {label_to_display}")
    axes[1].imshow(std_img, cmap='gray')

    if save_image:
      plt.savefig(f"{file_path}/avg_var_{label_to_display}.png", bbox_inches='tight', dpi=150)
    else:
      plt.tight_layout()
      plt.show()
      print("\n")

In [ ]:
plot_mean_variability_per_labels(X=X, y=y, figsize=(12,5),save_image=True)

---

## Difference between average benign and average malignant skin lesion images

In [ ]:
y_str = np.asarray(y, dtype=str)
label_names = np.unique(y_str)

# Try to find labels that match "benign" and "malignant"
benign_candidates = [label for label in label_names if "benign" in label.lower()]
malignant_candidates = [
    label for label in label_names
    if ("malign" in label.lower()) or ("melanoma" in label.lower()) or ("cancer" in label.lower())
]

if not benign_candidates:
    benign_candidates = [str(label_names[0])]

if not malignant_candidates:
    if len(label_names) > 1:
        malignant_candidates = [str(label_names[1])]
    else:
        malignant_candidates = [str(label_names[0])]

benign_label = benign_candidates[0]
malignant_label = malignant_candidates[0]

benign_mask = (y_str == benign_label)
malignant_mask = (y_str == malignant_label)

avg_benign = X[benign_mask].mean(axis=0)
avg_malignant = X[malignant_mask].mean(axis=0)

# Difference map: positive values mean more in benign, negative mean more in malignant
difference_map = avg_benign - avg_malignant
abs_difference_map = np.abs(difference_map)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(avg_benign)
axes[0].set_title(f"Average {benign_label}")
axes[0].axis("off")

axes[1].imshow(avg_malignant)
axes[1].set_title(f"Average {malignant_label}")
axes[1].axis("off")

axes[2].imshow(abs_difference_map, cmap="coolwarm")
axes[2].set_title(f"Absolute Difference\n({benign_label} - {malignant_label})")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(avg_benign)
axes[0].set_title(f"Average {benign_label}")
axes[0].axis("off")

axes[1].imshow(avg_malignant)
axes[1].set_title(f"Average {malignant_label}")
axes[1].axis("off")

axes[2].imshow(abs_difference_map, cmap="coolwarm")
axes[2].set_title(f"Absolute Difference\n({benign_label} - {malignant_label})")
axes[2].axis("off")

plt.tight_layout()
plt.savefig(f"outputs/v1/benign_vs_malignant_difference.png", bbox_inches="tight", dpi=150)
plt.show()

And now we have finally saved the average differences between malignant, benign, and the absolute difference between them!

---

## Image Montage

Here we will create two image montages. One for malignant and one for benign

In [ ]:
y_str = np.asarray(y, dtype=str)
label_names = np.unique(y_str)

benign_candidates = [label for label in label_names if "benign" in label.lower()]
malignant_candidates = [
    label for label in label_names
    if ("malign" in label.lower()) or ("melanoma" in label.lower()) or ("cancer" in label.lower())
]

if not benign_candidates:
    benign_candidates = [str(label_names[0])]
if not malignant_candidates:
    if len(label_names) > 1:
        malignant_candidates = [str(label_names[1])]
    else:
        malignant_candidates = [str(label_names[0])]

benign_label = benign_candidates[0]
malignant_label = malignant_candidates[0]

benign_images = X[y_str == benign_label][:30]
malignant_images = X[y_str == malignant_label][:30]

def create_montage(images, title, rows=5, cols=6):
    fig, axes = plt.subplots(rows, cols, figsize=(15, 12))
    fig.suptitle(title, fontsize=16)

    for ax in axes.flat:
        ax.axis("off")

    for i, ax in enumerate(axes.flat):
        if i < len(images):
            img = images[i]
            # Ensure values are in range 0..1 for display
            img = np.clip(img, 0, 1)
            ax.imshow(img)
        else:
            ax.axis("off")

    plt.tight_layout()
    return fig

benign_fig = create_montage(benign_images, f"Benign Lesion Montage - {benign_label}")
malignant_fig = create_montage(malignant_images, f"Malignant Lesion Montage - {malignant_label}")

benign_fig.savefig(f"{file_path}/benign_montage.png", bbox_inches="tight", dpi=150)
malignant_fig.savefig(f"{file_path}/malignant_montage.png", bbox_inches="tight", dpi=150)

plt.show()

And now we save the images

In [20]:
benign_fig = create_montage(benign_images, f"Benign Lesion Montage - {benign_label}")
malignant_fig = create_montage(malignant_images, f"Malignant Lesion Montage - {malignant_label}")

benign_fig.savefig(f"{file_path}/benign_montage.png", bbox_inches="tight", dpi=150)
malignant_fig.savefig(f"{file_path}/malignant_montage.png", bbox_inches="tight", dpi=150)

print(f"Saved images to: {file_path}")
plt.show()

---

# Conclusions and Next Steps

We have concluded Hypothesis 2(there is a difference between malignant and benign skin lesions).

Next, we will develop the CNN(ML Model) to predict benigs malignant skin lesions

Lastly, do the following commands in the terminal to save the files

git add .

git commit -m "message you want to add"

git push